# `ro_calculation` — worked examples

How to call the package. Each section maps to a section of `README.md`.

Run from `notebooks/`, with the package installed (`pip install -e ..`).

In [ ]:
import os
from datetime import date
from pathlib import Path

import pandas as pd

# Market data (futures.csv, the PFC, holidays.csv) is read from the data/ folder of the
# current working directory. A notebook runs in notebooks/, so point RO_DATA_DIR at ../data.
os.environ["RO_DATA_DIR"] = str((Path("..") / "data").resolve())

from ro_calculation import load_price_frames, run_asof

pd.set_option("display.float_format", "{:.2f}".format)

TARGET_QUARTER = "q3_26"
ASOF = date(2026, 6, 30)

df_power, df_gas, df_eua = load_price_frames()   # reads ../data/futures.csv
df_power.head()

## 1. One as-of, one quarter

`run_asof` returns a dict with six keys.

In [ ]:
result = run_asof(ASOF, df_power, df_gas, df_eua, TARGET_QUARTER)

print(result.keys())
print(result["run_as_of"], result["run_as_of_status"], "realised:", result["realised"])
result["ro_scenarios"]

`legs` — which exchange supplied each leg, and how many trading days remain in its
averaging window.

In [ ]:
result["legs"]

`df` — one row per leg per commodity, a `total` row per commodity, and a final `RO` row.
See the *`result["df"]`* table in `README.md` for the column meanings.

In [ ]:
result["df"]

In [ ]:
# The RO row alone.
result["df"][result["df"]["commodity"] == "RO"]

## 2. Several quarters, or several as-of dates, in one call

Either `asof` or `target_quarter` (or both) accepts a list. One list returns a dict keyed
by that argument; two lists return a dict keyed by `asof`, each value keyed by quarter.

In [ ]:
QUARTERS = ["q2_27", "q3_27", "q1_28", "q2_28"]

multi_quarter = run_asof(date(2026, 7, 23), df_power, df_gas, df_eua, QUARTERS)
pd.DataFrame({q: multi_quarter[q]["ro_scenarios"] for q in QUARTERS}).T

In [ ]:
ASOFS = [date(2026, 6, 30), date(2026, 7, 1), date(2026, 7, 15)]

multi_asof = run_asof(ASOFS, df_power, df_gas, df_eua, "q2_27")
pd.DataFrame({a: multi_asof[a]["ro_scenarios"] for a in ASOFS}).T

In [ ]:
# Both as lists: keyed by as-of, then by quarter.
grid = run_asof(ASOFS, df_power, df_gas, df_eua, QUARTERS)
pd.DataFrame({a: {q: grid[a][q]["ro_scenarios"]["reference"] for q in QUARTERS} for a in ASOFS}).T

## 3. `use_pfc=True` — floating portion from the hourly PFC

The floating (still-open) portion of each leg comes from the PFC instead of the last
futures close held flat; the fixed portion is futures-sourced either way. Legs the PFC
contributed to carry `"PFC"` in `price_source`.

In [ ]:
PFC_ASOF = date(2026, 7, 24)
PFC_QUARTER = "q2_27"

futures_only = run_asof(PFC_ASOF, df_power, df_gas, df_eua, PFC_QUARTER, use_pfc=False)
with_pfc = run_asof(PFC_ASOF, df_power, df_gas, df_eua, PFC_QUARTER, use_pfc=True)

cols = ["strip", "fixed", "float", "reference", "price_source"]
with_pfc["df"][with_pfc["df"]["commodity"] == "power"][cols].merge(
    futures_only["df"][futures_only["df"]["commodity"] == "power"][cols],
    on="strip", suffixes=(" (use_pfc)", " (futures only)"),
)

`use_pfc=True` also resolves legs that have no futures quote at all on their own delivery
window, which is what makes far-dated quarters computable. Every quarter below raises
`ValueError` (no EUA quote that far out) when called with `use_pfc=False`.

In [ ]:
FAR_QUARTERS = ["q3_28", "q1_29", "q2_29", "q3_29", "q1_30", "q2_30", "q3_30", "q1_31", "q2_31", "q3_31"]

full_df = pd.concat(
    [run_asof(PFC_ASOF, df_power, df_gas, df_eua, q, use_pfc=True)["df"] for q in FAR_QUARTERS],
    ignore_index=True,
)
full_df[full_df["commodity"] == "EUA"][["quarter", "fixed", "float", "reference", "price_source"]]

In [ ]:
# One RO per quarter.
full_df[full_df["commodity"] == "RO"][["quarter", "reference", "pct_open", "price_source"]]

## 4. Realised quarters

When the quarter's RO has already been published and `data/ro_realised.xlsx` has a row for
it, that figure is returned verbatim: `realised=True`, `legs` empty, `df` holding only the
`RO` row, and all three scenarios equal. Standing on an earlier `asof`, the same quarter is
calculated as usual.

In [ ]:
published = run_asof(date(2026, 8, 27), df_power, df_gas, df_eua, "q3_26")
estimated = run_asof(date(2026, 6, 30), df_power, df_gas, df_eua, "q3_26")

print("realised:", published["realised"], published["ro_scenarios"])
print("estimated:", estimated["realised"], estimated["ro_scenarios"])
published["df"]

## 5. Constants

`constants_for` builds the mapping `run_asof` uses when `constants` is left as `None`.
Pass your own mapping to override it.

In [ ]:
from ro_calculation import constants_for, installation_parameters, ivpee_for_quarter

pd.Series(constants_for(TARGET_QUARTER))

In [ ]:
print(ivpee_for_quarter(TARGET_QUARTER), "%")
pd.Series(installation_parameters(2026))

In [ ]:
# An explicit mapping is used as-is.
custom = {**constants_for(TARGET_QUARTER), "IVPEE": 0.0}
run_asof(ASOF, df_power, df_gas, df_eua, TARGET_QUARTER, constants=custom)["ro_scenarios"]

## 6. `calculate_ro` on its own

`formula.py` imports nothing from the rest of the package: `calculate_ro(prices, constants)`
takes a mapping with `P_m`, `P_pvb`, `P_co2` plus a constant mapping and returns the RO in
€/MWhE. `REQUIRED_CONSTANTS` lists the constant keys it needs.

In [ ]:
from ro_calculation import REQUIRED_CONSTANTS, calculate_ro

print(sorted(REQUIRED_CONSTANTS))
prices = {"P_m": 90.0, "P_pvb": 45.0, "P_co2": 80.0}
calculate_ro(prices, constants_for(TARGET_QUARTER))

## 7. Windows

`averaging_window` gives the dates a leg is averaged over; `delivery_window` the dates it
is delivered over.

In [ ]:
from ro_calculation import averaging_window, delivery_window, parse_target_quarter

print(parse_target_quarter(TARGET_QUARTER))
pd.DataFrame(
    [
        {
            "leg": leg,
            "averaging (power)": averaging_window(TARGET_QUARTER, "power", leg),
            "delivery": delivery_window(TARGET_QUARTER, leg),
        }
        for leg in ["Y", "Q", "M1", "M2", "M3"]
    ]
)

## 8. Writing the output workbook

`run` does the same calculation and writes a filled copy of
the workbook template shipped in the package into `output/`. Equivalent on the command line:

```bash
python -m ro_calculation q2_27 --asof 2026-07-24 --use-pfc
```

In [ ]:
from ro_calculation import run

run("q2_27", asof=PFC_ASOF, use_pfc=True, out_dir=Path("..") / "output")